<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# Logistic Regression with Python


Estimated time needed: **25** minutes
    

## Objetivos

Después de completar este laboratorio, podrá:

* Utilizar la regresión logística de Scikit para clasificar
* Comprender la matriz de confusión


En este cuaderno, aprenderá regresión logística y luego creará un modelo para una empresa de telecomunicaciones, para predecir cuándo sus clientes se irán a un competidor, de modo que puedan tomar algunas medidas para retenerlos.


<h1>Table of contents</h1>

<div class="alert alert-block alert-info" style="margin-top: 20px">
    <ol>
        <li><a href="#about_dataset">About the dataset</a></li>
        <li><a href="#preprocessing">Data pre-processing and selection</a></li>
        <li><a href="#modeling">Modeling (Logistic Regression with Scikit-learn)</a></li>
        <li><a href="#evaluation">Evaluation</a></li>
        <li><a href="#practice">Practice</a></li>
    </ol>
</div>
<br>
<hr>


<a id="ref1"></a>
## ¿Cuál es la diferencia entre la regresión lineal y la regresión logística?

Si bien la regresión lineal es adecuada para estimar valores continuos (por ejemplo, estimar el precio de una casa), no es la mejor herramienta para predecir la clase de un punto de datos observado. Para estimar la clase de un punto de datos, necesitamos algún tipo de orientación sobre cuál sería la <b>clase más probable</b> para ese punto de datos. Para esto, utilizamos la <b>regresión logística</b>.

<font size = 3><strong>Recuerde la regresión lineal:</strong></font>
<br>
<br>
Como sabe, la <b>regresión lineal</b> encuentra una función que relaciona una variable dependiente continua, <b>y</b>, con algunos predictores (variables independientes $x_1$, $x_2$, etc.). Por ejemplo, la regresión lineal simple supone una función de la forma:
<br><br>
$$
y = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \cdots
$$
<br>
y encuentra los valores de los parámetros $\theta_0$, $\theta_1$, $\theta_2$, etc., donde el término $\theta_0$ es la "intersección". En general, se puede mostrar como:
<br><br>

$$
ℎ_\theta(𝑥) = \theta^TX
$$
<p></p>

La regresión logística es una variación de la regresión lineal, que se utiliza cuando la variable dependiente observada, <b>y</b>, es categórica. Produce una fórmula que predice la probabilidad de la etiqueta de clase como una función de las variables independientes.

La regresión logística se ajusta a una curva especial en forma de s tomando la función de regresión lineal y transformando la estimación numérica en una probabilidad con la siguiente función, que se llama función sigmoidea 𝜎:

$$
ℎ_\theta(𝑥) = \sigma({\theta^TX}) = \frac {e^{(\theta_0 + \theta_1 x_1 + \theta_2 x_2 +...)}}{1 + e^{(\theta_0 + \theta_1 x_1 + \theta_2 x_2 +\cdots)}}
$$
O:
$$
ProbabilidadDeUnaClase_1 = P(Y=1|X) = \sigma({\theta^TX}) = \frac{e^{\theta^TX}}{1+e^{\theta^TX}}
$$

En esta ecuación, ${\theta^TX}$ es el resultado de la regresión (la suma de las variables ponderadas por los coeficientes), `exp` es la función exponencial y $\sigma(\theta^TX)$ es la función sigmoidea o [logística [función](http://en.wikipedia.org/wiki/Logistic_function), también llamada curva logística. Tiene forma de "S" común (curva sigmoidea).

En resumen, la regresión logística pasa la entrada a través de la función logística/sigmoidea, pero luego trata el resultado como una probabilidad:

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/images/mod_ID_24_final.png" width="400" align="center">

El objetivo del algoritmo __Regresión logística__ es encontrar los mejores parámetros θ, para $ℎ_\theta(𝑥)$ = $\sigma({\theta^TX})$, de tal manera que el modelo prediga mejor la clase de cada caso.


### Pérdida de clientes con regresión logística
Una empresa de telecomunicaciones está preocupada por la cantidad de clientes que abandonan su negocio de telefonía fija para pasarse a la competencia de cable. Necesitan saber quiénes se van. Imagine que usted es un analista de esta empresa y tiene que averiguar quiénes se van y por qué.


In [ ]:
#!pip install scikit-learn==0.23.1
!pip install scikit-learn
!pip install matplotlib
!pip install pandas 
!pip install numpy 
%matplotlib inline

Let's first import required libraries:


In [ ]:
import pandas as pd
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
%matplotlib inline 
import matplotlib.pyplot as plt

<h2 id="about_dataset">Acerca del conjunto de datos</h2>
Utilizaremos un conjunto de datos de telecomunicaciones para predecir la pérdida de clientes. Se trata de un conjunto de datos históricos de clientes en el que cada fila representa un cliente. Los datos son relativamente fáciles de entender y puede descubrir información que puede utilizar de inmediato. Por lo general, es menos costoso conservar a los clientes que adquirir nuevos, por lo que el objetivo de este análisis es predecir los clientes que se quedarán en la empresa.

Este conjunto de datos proporciona información que le ayudará a predecir qué comportamiento le ayudará a retener a los clientes. Puede analizar todos los datos relevantes de los clientes y desarrollar programas de retención de clientes específicos.

El conjunto de datos incluye información sobre:

- Clientes que se dieron de baja en el último mes: la columna se llama Churn
- Servicios a los que se ha suscrito cada cliente: teléfono, líneas múltiples, Internet, seguridad en línea, copia de seguridad en línea, protección de dispositivos, soporte técnico y transmisión de TV y películas
- Información de la cuenta del cliente: cuánto tiempo ha sido cliente, contrato, método de pago, facturación electrónica, cargos mensuales y cargos totales
- Información demográfica sobre los clientes: género, rango de edad y si tienen pareja y dependientes

###  Load the Telco Churn data 
Telco Churn is a hypothetical data file that concerns a telecommunications company's efforts to reduce turnover in its customer base. Each case corresponds to a separate customer and it records various demographic and service usage information. Before you can work with the data, you must use the URL to get the ChurnData.csv.

To download the data, we will use `!wget` to download it from IBM Object Storage.


In [ ]:
#Click here and press Shift+Enter
!wget -O ChurnData.csv https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-ML0101EN-SkillsNetwork/labs/Module%203/data/ChurnData.csv

## Load Data From CSV File  


In [ ]:
churn_df = pd.read_csv("ChurnData.csv")
churn_df.head()

<h2 id="preprocessing">Data pre-processing and selection</h2>


Seleccionemos algunas características para el modelado. Además, cambiamos el tipo de datos de destino para que sea un número entero, ya que es un requisito del algoritmo Skitlearn:

In [ ]:
churn_df = churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip',   'callcard', 'wireless','churn']]
churn_df['churn'] = churn_df['churn'].astype('int')
churn_df.head()

## Práctica
¿Cuántas filas y columnas hay en total en este conjunto de datos? ¿Cuáles son los nombres de las columnas?


In [ ]:
# write your code here


<details><summary>Click here for the solution</summary>

```python
churn_df.shape

```

</details>



Let's define X, and y for our dataset:


In [ ]:
X = np.asarray(churn_df[['tenure', 'age', 'address', 'income', 'ed', 'employ', 'equip']])
X[0:5]

In [ ]:
y = np.asarray(churn_df['churn'])
y [0:5]

Also, we normalize the dataset:


In [ ]:
from sklearn import preprocessing
X = preprocessing.StandardScaler().fit(X).transform(X)
X[0:5]

## Train/Test dataset


We split our dataset into train and test set:


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, random_state=4)
print ('Train set:', X_train.shape,  y_train.shape)
print ('Test set:', X_test.shape,  y_test.shape)

<h2 id="modeling">Modeling (Logistic Regression with Scikit-learn)</h2>


Construyamos nuestro modelo usando __LogisticRegression__ del paquete Scikit-learn. Esta función implementa la regresión logística y puede usar diferentes optimizadores numéricos para encontrar parámetros, incluidos los solucionadores 'newton-cg', 'lbfgs', 'liblinear', 'sag' y 'saga'. Puede encontrar información extensa sobre los pros y los contras de estos optimizadores si la busca en Internet.

La versión de Regresión logística en Scikit-learn admite la regularización. La regularización es una técnica utilizada para resolver el problema de sobreajuste de los modelos de aprendizaje automático.
El parámetro __C__ indica __inverse of regularization strength__ que debe ser un punto flotante positivo. Los valores más pequeños especifican una regularización más fuerte.
Ahora ajustemos nuestro modelo con el conjunto de entrenamiento:

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix
LR = LogisticRegression(C=0.01, solver='liblinear').fit(X_train,y_train)
LR

Ahora podemos predecir usando nuestro conjunto de pruebas:


In [ ]:
yhat = LR.predict(X_test)
yhat

__predict_proba__ devuelve estimaciones para todas las clases, ordenadas por la etiqueta de las clases. Por lo tanto, la primera columna es la probabilidad de la clase 0, P(Y=0|X), y la segunda columna es la probabilidad de la clase 1, P(Y=1|X):


In [ ]:
yhat_prob = LR.predict_proba(X_test)
yhat_prob

<h2 id="evaluation">Evaluation</h2>


### jaccard index
Probemos el índice Jaccard para evaluar la precisión. Podemos definir Jaccard como el tamaño de la intersección dividido por el tamaño de la unión de los dos conjuntos de etiquetas. Si todo el conjunto de etiquetas predichas para una muestra coincide estrictamente con el conjunto real de etiquetas, entonces la precisión del subconjunto es 1,0; de lo contrario, es 0,0.



In [ ]:
from sklearn.metrics import jaccard_score
jaccard_score(y_test, yhat,pos_label=0)

### confusion matrix
Otra forma de evaluar la precisión del clasificador es mirar la __matriz de confusión__.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')
print(confusion_matrix(y_test, yhat, labels=[1,0]))

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test, yhat, labels=[1,0])
np.set_printoptions(precision=2)


# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['churn=1','churn=0'],normalize= False,  title='Confusion matrix')

Veamos la primera fila. La primera fila corresponde a los clientes cuyo valor de abandono real en el conjunto de prueba es 1.
Como puede calcular, de 40 clientes, el valor de abandono de 15 de ellos es 1.
De estos 15 casos, el clasificador predijo correctamente 6 de ellos como 1 y 9 de ellos como 0.

Esto significa que, para 6 clientes, el valor de abandono real fue 1 en el conjunto de prueba y el clasificador también predijo correctamente esos como 1. Sin embargo, mientras que la etiqueta real de 9 clientes fue 1, el clasificador los predijo como 0, lo que no es muy bueno. Podemos considerarlo como el error del modelo para la primera fila.

¿Qué sucede con los clientes con un valor de abandono de 0? Veamos la segunda fila.
Parece que había 25 clientes cuyo valor de abandono era 0.

El clasificador predijo correctamente 24 de ellos como 0 y uno de ellos de forma incorrecta como 1. Por lo tanto, ha hecho un buen trabajo al predecir los clientes con un valor de abandono 0. Una ventaja de la matriz de confusión es que muestra la capacidad del modelo para predecir o separar correctamente las clases. En un caso específico del clasificador binario, como este ejemplo, podemos interpretar estos números como el recuento de verdaderos positivos, falsos positivos, verdaderos negativos y falsos negativos.


In [ ]:
print (classification_report(y_test, yhat))


En función del recuento de cada sección, podemos calcular la precisión y la recuperación de cada etiqueta:

- __Precision__ es una medida de la precisión siempre que se haya predicho una etiqueta de clase. Se define por: precisión = TP / (TP + FP)

- __Recall__ es la tasa de verdaderos positivos. Se define como: Recuperación =  TP / (TP + FN)

Por lo tanto, podemos calcular la precisión y la recuperación de cada clase.

__F1-score:__
Ahora estamos en condiciones de calcular las puntuaciones F1 para cada etiqueta en función de la precisión y la recuperación de esa etiqueta.

La puntuación F1 es el promedio armónico de la precisión y la recuperación, donde una puntuación F1 alcanza su mejor valor en 1 (precisión y recuperación perfectas) y el peor en 0. Es una buena forma de demostrar que un clasificador tiene un buen valor tanto para la recuperación como para la precisión.

Finalmente, podemos decir que la precisión promedio de este clasificador es el promedio del puntaje F1 para ambas etiquetas, que es 0,72 en nuestro caso.


### log loss
Ahora, probemos __log loss__ para la evaluación. En la regresión logística, el resultado puede ser la probabilidad de que el cliente abandone el sistema si es sí (o igual a 1). Esta probabilidad es un valor entre 0 y 1.
Log loss (pérdida logarítmica) mide el rendimiento de un clasificador, donde el resultado previsto es un valor de probabilidad entre 0 y 1.


In [ ]:
from sklearn.metrics import log_loss
log_loss(y_test, yhat_prob)

<h2 id="practice">Practice</h2>
Intente crear un modelo de regresión logística nuevamente para el mismo conjunto de datos, pero esta vez, use valores diferentes de __solver__ y __regularization__. ¿Cuál es el nuevo valor de __logLoss__?


In [ ]:
# write your code here



<details><summary>Click here for the solution</summary>

```python
LR2 = LogisticRegression(C=0.01, solver='sag').fit(X_train,y_train)
yhat_prob2 = LR2.predict_proba(X_test)
print ("LogLoss: : %.2f" % log_loss(y_test, yhat_prob2))

```

</details>



### Thank you for completing this lab!


## Author

Saeed Aghabozorgi


### Other Contributors

<a href="https://www.linkedin.com/in/joseph-s-50398b136/" target="_blank">Joseph Santarcangelo</a>

## <h3 align="center"> © IBM Corporation 2020. All rights reserved. <h3/>

<!--

## Change Log


|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2021-01-21  | 2.2  | Lakshmi  |  Updated sklearn library|
| 2020-11-03  | 2.1  | Lakshmi  |  Updated URL of csv |
| 2020-08-27  | 2.0  | Lavanya  |  Moved lab to course repo in GitLab |
|   |   |   |   |
|   |   |   |   |

--!>


